[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# asyncpg


## What you will be able to do

Write queries for asyncpg: `$1` for a parameter, `fetch`, `fetchrow`, `fetchval` and `execute` for
the four shapes of answer, and `Record` for the row that comes back. Say what asyncpg does with
transactions, which is the one thing it does differently from psycopg and the one thing that
surprises people: every statement is committed as it returns until you open a block. Open that
block, nest it, and set its isolation level. Put a time limit on a single call. And read a query
written for psycopg closely enough to port it without guessing.


## The idea

### The problem

asyncpg is a second driver for the same database, and it is not a variation on the first. It is not
a DB-API driver, it has no synchronous half, it speaks the PostgreSQL binary protocol itself rather
than through libpq, and it spells almost everything differently. Code pasted across from psycopg
does not run, and the first two things that stop it are cosmetic enough to be fixed in a minute.

The third is not cosmetic. psycopg opens a transaction for you and waits for a commit.
asyncpg does not: a statement that runs outside a transaction block is finished and durable the
moment it returns. That is the opposite of what **Transactions and Errors** taught, and a program
ported without noticing is a program with no rollback.

### What asyncpg is for

Speed, mostly, and a smaller and more explicit API. It is PostgreSQL only, which is how it can
decode results without libpq and without a generic layer in between. **Which Driver** measures the
difference rather than asserting it.

### Why it works that way

asyncpg has no autocommit setting, because it has no implicit transaction to switch off. A
statement on its own is a transaction of one, which is what PostgreSQL does with any statement that
arrives outside a block. `async with conn.transaction()` is how you say the next several statements
belong together.

### Where this shows up

Anywhere someone reaches for the faster driver, and anywhere a codebase has both: a service using
asyncpg for its readers and psycopg for its loading, which is exactly what **An Event Store** builds
at the end of this guide.

### What this notebook covers

The four fetch methods and `executemany`. `Record`, beside the psycopg row you already know. The
transaction block, its savepoints and its isolation level. `timeout=`, which psycopg has no
per-query equivalent for. `copy_records_to_table`, cursors, and the type codec that a `jsonb` column
needs. Then the three ported-code failures and the one that does not raise at all.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio

import asyncpg


async def main():
    conn = await asyncpg.connect(database="guide")

    row = await conn.fetchrow(                              # a Record: a tuple and a mapping at once
        "SELECT kind, count(*) AS n FROM events GROUP BY kind ORDER BY n LIMIT 1")
    print(row)
    print("by name:", row["kind"], " by position:", row[1])

    rarest = await conn.fetchval(                           # $1, not %s
        "SELECT count(*) FROM events WHERE kind = $1", row["kind"])
    print("counted again:", rarest)

    await conn.execute("DROP TABLE IF EXISTS tally")
    await conn.execute("CREATE TABLE tally (kind text, n int)")
    await conn.execute("INSERT INTO tally VALUES ($1, $2)", row["kind"], rarest)

    seen_elsewhere = await asyncpg.connect(database="guide")
    print("another connection already sees the row:",       # nothing was committed, and yet
          await seen_elsewhere.fetchval("SELECT n FROM tally"))

    await conn.close()
    await seen_elsewhere.close()


asyncio.run(main())
```

```
<Record kind='click' n=1666>
by name: click  by position: 1666
counted again: 1666
another connection already sees the row: 1666
```

Three things to notice. The row prints as a `Record` and answers to both a name and a number. The
parameter is `$1`. And the last line is the surprise: a second connection can already see the
insert, on a program that never called `commit` and never opened a transaction.


## Setup

Eleven imports, both drivers, the server, and two helpers.

- `asyncpg` is this notebook's driver and `exceptions` is where its error classes live
- `psycopg` appears only to put the two side by side, since the point is porting between them
- `asyncio` runs the event loop, `json` decodes a `jsonb` column, and `time` measures one wait
- `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and `PackageNotFoundError`

`fresh` empties the `tally` table so the sections below can be run in any order, and `tally` reads it
back through whichever connection is asking, which is how the transaction sections tell what another
session can see.


In [1]:
import asyncio
import getpass
import json
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from asyncpg import exceptions

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

async def fresh(conn):
    """An empty tally table, so the sections below can be run in any order."""
    await conn.execute("DROP TABLE IF EXISTS tally")
    await conn.execute("CREATE TABLE tally (kind text, n int)")


async def tally(conn):
    """What is in it, read through whichever connection is asking."""
    return [tuple(row) for row in await conn.fetch("SELECT kind, n FROM tally ORDER BY kind")]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


## Worked examples

### The four ways to ask

psycopg gives you a cursor and you fetch from it. asyncpg gives you the answer, in whichever of four
shapes you asked for:


In [2]:
conn = await asyncpg.connect(database="guide")

print("fetch     ->", await conn.fetch("SELECT id, kind FROM events ORDER BY id LIMIT 2"))
print("fetchrow  ->", await conn.fetchrow("SELECT id, kind FROM events ORDER BY id LIMIT 1"))
print("fetchval  ->", await conn.fetchval("SELECT count(*) FROM events"))
await conn.execute("CREATE TEMP TABLE shapes (n int)")
print("execute   ->", repr(await conn.execute("INSERT INTO shapes VALUES (1), (2)")))


fetch     -> [<Record id=1 kind='view'>, <Record id=2 kind='purchase'>]
fetchrow  -> <Record id=1 kind='view'>
fetchval  -> 5000
execute   -> 'INSERT 0 2'


`fetch` gives a list, `fetchrow` gives one `Record` or `None`, `fetchval` gives one value or `None`,
and `execute` gives back the server's command tag as a string. Nothing here has a cursor in it.

`None` for an empty result is worth pausing on, because it is not an error:


In [3]:
print("fetchrow on no rows:", await conn.fetchrow("SELECT 1 WHERE false"))
print("fetchval on no rows:", await conn.fetchval("SELECT 1 WHERE false"))
print("fetchval on a NULL: ", await conn.fetchval("SELECT NULL::int"))


fetchrow on no rows: None
fetchval on no rows: None
fetchval on a NULL:  None


The last two are the same `None` from different causes, which is a real ambiguity and a reason to
prefer `fetchrow` when "no row" and "a null" mean different things to your program.

### A Record, beside a psycopg row

Same query, both drivers, so the difference is visible rather than described:


In [4]:
sql = "SELECT id, kind FROM events ORDER BY id LIMIT 1"

record = await conn.fetchrow(sql)
print("asyncpg:", repr(record))
print("  by name:", record["kind"], "| by position:", record[1], "| as a dict:", dict(record))

with psycopg.connect("dbname=guide") as sync:
    print("psycopg:", repr(sync.execute(sql).fetchone()))
    with sync.cursor(row_factory=psycopg.rows.dict_row) as cur:
        print("  with a row factory:", cur.execute(sql).fetchone())


asyncpg: <Record id=1 kind='view'>
  by name: view | by position: view | as a dict: {'id': 1, 'kind': 'view'}
psycopg: (1, 'view')
  with a row factory: {'id': 1, 'kind': 'view'}


psycopg gives a plain tuple and lets you choose something else with a row factory. asyncpg gives a
`Record`, always, and a `Record` is already both: indexable like a tuple, subscriptable by column
name, and convertible with `dict`. There is no factory to choose because there is nothing to choose
between.

It is read-only, which catches people who expected a dictionary:


In [5]:
try:
    record["kind"] = "changed"
except TypeError as error:
    print("TypeError:", error)

changeable = dict(record)                                           # a copy you own
changeable["kind"] = "changed"
print("a dict made from it:", changeable)


TypeError: 'asyncpg.protocol.record.Record' object does not support item assignment
a dict made from it: {'id': 1, 'kind': 'changed'}


### Writing, and the transaction that was not there

This is the section the notebook exists for. Two statements, the second one broken:


In [6]:
await fresh(conn)
watcher = await asyncpg.connect(database="guide")                   # a second session, watching

await conn.execute("INSERT INTO tally VALUES ($1, $2)", "click", 1666)
try:
    await conn.execute("INSERT INTO tally VALUES ($1, $2)", "view", "not a number")
except exceptions.DataError as error:
    print("the second insert failed:", type(error).__name__)

print("what the other session sees:", await tally(watcher))


the second insert failed: DataError
what the other session sees: [('click', 1666)]


The first row is there, committed, permanently, and there is nothing to roll back because there was
never a transaction. In psycopg both statements would have been inside one, and the failure would
have left the whole thing to be rolled back.

The fix is to say what belongs together:


In [7]:
await fresh(conn)

try:
    async with conn.transaction():
        await conn.execute("INSERT INTO tally VALUES ($1, $2)", "click", 1666)
        await conn.execute("INSERT INTO tally VALUES ($1, $2)", "view", "not a number")
except exceptions.DataError as error:
    print("the block failed:", type(error).__name__)

print("what the other session sees:", await tally(watcher))


the block failed: DataError
what the other session sees: []


Nothing. Leaving the block by an exception rolls it back, leaving it normally commits it, and that
is the whole contract. There is no `conn.commit()` in asyncpg because there is nothing to commit
outside a block and the block does it for you.

### Nesting, which is savepoints

A transaction inside a transaction is a savepoint, and it can fail on its own:


In [8]:
await fresh(conn)

async with conn.transaction():
    await conn.execute("INSERT INTO tally VALUES ($1, $2)", "kept", 1)

    try:
        async with conn.transaction():                              # a savepoint
            await conn.execute("INSERT INTO tally VALUES ($1, $2)", "dropped", 2)
            raise RuntimeError("the optional part did not work out")
    except RuntimeError as error:
        print("the inner block rolled back:", error)

    await conn.execute("INSERT INTO tally VALUES ($1, $2)", "also kept", 3)

print("committed:", await tally(watcher))


the inner block rolled back: the optional part did not work out
committed: [('also kept', 3), ('kept', 1)]


The same shape as psycopg's nested `with conn.transaction()`, and for the same reason: an inner block
that fails does not have to take the outer one down with it. **Transactions and Errors** made the
argument, and asyncpg spells it identically.

The isolation level is an argument on the block:


In [9]:
async with conn.transaction(isolation="serializable"):
    print("inside the block: ", await conn.fetchval("SHOW transaction_isolation"))

print("outside the block:", await conn.fetchval("SHOW transaction_isolation"))
print("the accepted values are read_uncommitted, read_committed, repeatable_read, serializable")


inside the block:  serializable
outside the block: read committed
the accepted values are read_uncommitted, read_committed, repeatable_read, serializable


Underscores, not spaces, and anything else is a `ValueError` before a statement is sent.

### A time limit on one call

Every one of the fetch methods takes `timeout=`, in seconds:


In [10]:
start = time.perf_counter()
try:
    await conn.fetchval("SELECT pg_sleep(30)", timeout=0.5)
except TimeoutError as error:
    print("TimeoutError, with no message at all:", repr(str(error)))

print(f"it gave up after {time.perf_counter() - start:.1f}s")
print("the connection is still usable:", await conn.fetchval("SELECT 1"))


TimeoutError, with no message at all: ''
it gave up after 0.5s
the connection is still usable: 1


That is a plain built-in `TimeoutError` carrying no message, which is easy to misread in a log. Note
what it is not: psycopg has no per-call equivalent, only `SET statement_timeout`, which applies to
every statement on that session until you change it back. A per-call limit is a real advantage of
this driver, and **Which Driver** counts it as one.

### Many rows in, many rows out

`executemany` for a handful, `copy_records_to_table` for a lot:


In [11]:
await fresh(conn)

await conn.executemany("INSERT INTO tally VALUES ($1, $2)",
                       [("click", 1), ("view", 2), ("purchase", 3)])
print("after executemany:", await tally(conn))

loaded = await conn.copy_records_to_table(
    "tally", records=[(f"kind {n}", n) for n in range(1000)], columns=["kind", "n"])
print("copy_records_to_table returned:", repr(loaded))
print("rows now:", await conn.fetchval("SELECT count(*) FROM tally"))


after executemany: [('click', 1), ('purchase', 3), ('view', 2)]
copy_records_to_table returned: 'COPY 1000'
rows now: 1003


`executemany` returns nothing, which is a difference from psycopg's `cur.rowcount`, and
`copy_records_to_table` returns the command tag. This is the same `COPY` machinery **COPY** measured,
reached from Python objects rather than from a file or a generator.

Reading a large result without holding it all is a cursor, and a cursor needs a transaction:


In [12]:
async with conn.transaction():                                      # required, and the next cell says why
    seen = []
    async for record in conn.cursor("SELECT id FROM events ORDER BY id"):
        seen.append(record["id"])
        if len(seen) == 5:
            break

print("first five, without loading five thousand:", seen)


first five, without loading five thousand: [1, 2, 3, 4, 5]


### A jsonb column arrives as text

asyncpg decodes what it knows, and it does not assume you want `json` parsed:


In [13]:
raw = await conn.fetchval("SELECT payload FROM events ORDER BY id LIMIT 1")
print("without a codec:", type(raw).__name__, raw)

await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")

decoded = await conn.fetchval("SELECT payload FROM events ORDER BY id LIMIT 1")
print("with a codec:   ", type(decoded).__name__, decoded, "| size:", decoded["size"])


without a codec: str {"n": 1, "size": 2}
with a codec:    dict {'n': 1, 'size': 2} | size: 2


psycopg gives you a `dict` for a `jsonb` column with no setup, which **JSONB** relied on. asyncpg
gives you the text and a hook to register once per connection. That is the trade the whole driver
makes: fewer assumptions, more to say out loud.

### When to reach for which

| What you want | asyncpg | psycopg |
|---|---|---|
| a parameter | `$1`, `$2` | `%s` |
| every row | `await conn.fetch(sql)` | `cur.execute(sql).fetchall()` |
| one row | `await conn.fetchrow(sql)` | `cur.execute(sql).fetchone()` |
| one value | `await conn.fetchval(sql)` | `cur.execute(sql).fetchone()[0]` |
| a write | `await conn.execute(sql)` | `cur.execute(sql)`, then `conn.commit()` |
| a row by name | `record["kind"]`, always | a `dict_row` factory |
| statements that belong together | `async with conn.transaction()` | the default, plus `commit` |
| one statement on its own | already committed | inside a transaction until you commit |
| a savepoint | a nested `conn.transaction()` | a nested `conn.transaction()` |
| a time limit on one call | `timeout=0.5` | `SET statement_timeout` for the session |
| a `jsonb` column as a `dict` | `set_type_codec` once | the default |

The default when you are porting is to assume nothing carries over except the SQL. The one line to
keep in mind is the write: in psycopg a statement is provisional until you commit, and in asyncpg it
is final unless you opened a block.

### Porting one function, finished

A psycopg function and the asyncpg version of it, running one after the other on the same data, so
that every line of the table above is doing something.


In [14]:
def summarize_with_psycopg(url):
    """Count the events of each kind, and record the busiest, as psycopg would."""
    with psycopg.connect(url) as conn:                              # a transaction is already open
        with conn.cursor(row_factory=psycopg.rows.dict_row) as cur:
            rows = cur.execute("SELECT kind, count(*) AS n FROM events "
                               "GROUP BY kind ORDER BY n DESC, kind").fetchall()
            cur.execute("INSERT INTO tally VALUES (%s, %s)", (rows[0]["kind"], rows[0]["n"]))
        conn.commit()                                               # without this, nothing happened
    return rows[0]["kind"], rows[0]["n"]


async def summarize_with_asyncpg(conn):
    """The same thing, ported: $1, Record, and a block around the write."""
    rows = await conn.fetch("SELECT kind, count(*) AS n FROM events "
                            "GROUP BY kind ORDER BY n DESC, kind")
    async with conn.transaction():                                  # without this, no rollback
        await conn.execute("INSERT INTO tally VALUES ($1, $2)", rows[0]["kind"], rows[0]["n"])
    return rows[0]["kind"], rows[0]["n"]


await fresh(conn)
print("psycopg:", summarize_with_psycopg("dbname=guide"))
print("asyncpg:", await summarize_with_asyncpg(conn))
print("both wrote the same row:", await tally(watcher))


psycopg: ('purchase', 1667)
asyncpg: ('purchase', 1667)
both wrote the same row: [('purchase', 1667), ('purchase', 1667)]


The two rows are identical, which is the point: the port answers the same question and writes the
same thing. Neither function is a translation of the other line by line, though. The `commit`
disappeared and a `transaction` block appeared in its place, the cursor and the row factory
disappeared because a `Record` already does that, and the placeholders changed.

### Where each part came from

| In the ported function | What it relies on | The section that showed it |
|---|---|---|
| `await conn.fetch(...)` | a list of `Record` | The four ways to ask |
| `rows[0]["kind"]` | a row that answers to a name | A Record, beside a psycopg row |
| `$1, $2` | asyncpg's placeholder | A first look |
| `async with conn.transaction()` | the write is final without it | Writing, and the transaction that was not there |
| psycopg's `conn.commit()` | the write is provisional until it | **Transactions and Errors** |
| `row_factory=dict_row` | psycopg's way to get names | **Connecting and Executing** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/11-asyncpg-solutions.ipynb).

**1.** Connect with asyncpg and print the same answer three ways, with `fetch`, `fetchrow` and
`fetchval`.


In [15]:
# your code here


**2.** Count the events of one kind, passing the kind as a parameter.


In [16]:
# your code here


**3.** Write a row without a transaction block, then show a second connection can see it.


In [17]:
# your code here


**4.** Do the same inside a block that raises, and show the second connection sees nothing.


In [18]:
# your code here


**5.** Give one query half a second and let it run out.


In [19]:
# your code here


**6.** Read a `jsonb` column as a `dict`.


In [20]:
# your code here


## Common errors

### asyncpg.exceptions.PostgresSyntaxError: syntax error at or near "%"


In [21]:
await conn.fetch("SELECT count(*) FROM events WHERE kind = %s", "click")


PostgresSyntaxError: syntax error at or near "%"

A query pasted over from psycopg. asyncpg does not use `%s` and does not rewrite anything before
sending it, so the `%` reaches the server and the server has no idea what it is.

The parameters are numbered, which means they can be repeated and reordered without passing a value
twice:


In [22]:
print(await conn.fetchval("SELECT count(*) FROM events WHERE kind = $1", "click"))
print(await conn.fetch("SELECT $1::text AS said, upper($1::text) AS shouted", "once"))


1666
[<Record said='once' shouted='ONCE'>]


### asyncpg.exceptions.DataError: invalid input for query argument $1


In [23]:
await conn.fetchval("SELECT count(*) FROM events WHERE id = $1", "5")


DataError: invalid input for query argument $1: '5' ('str' object cannot be interpreted as an integer)

The column is an integer and the value is a string. psycopg would have sent the text and let the
server cast it. asyncpg checks the type in Python against what the server said the parameter is, and
refuses before sending anything.

This is stricter, and it is the kind of strictness that finds a bug in the caller rather than in the
query. Convert on your side:


In [24]:
print("converted:", await conn.fetchval("SELECT count(*) FROM events WHERE id = $1", int("5")))

try:
    await conn.fetch("SELECT $1::int, $2::int", 1)                  # one value, two parameters
except exceptions.InterfaceError as error:
    print("a different mistake:", error)


converted: 1
a different mistake: the server expects 2 arguments for this query, 1 was passed
HINT:  Check the query against the passed list of arguments.


### asyncpg.exceptions.NoActiveSQLTransactionError: cursor cannot be created outside of a transaction


In [25]:
async for record in conn.cursor("SELECT id FROM events"):
    print(record)


NoActiveSQLTransactionError: cursor cannot be created outside of a transaction

A cursor is a thing the server holds open for you, and PostgreSQL only holds one inside a
transaction. psycopg hides this because it has a transaction open already. asyncpg does not have one
open, so it says so.

It is the same rule as the server-side cursor in **Server-Side Cursors**, arriving as an error
instead of silently working:


In [26]:
async with conn.transaction():
    async for record in conn.cursor("SELECT id FROM events ORDER BY id"):
        print("first id:", record["id"])
        break


first id: 1


### No error: the write that a failing program keeps


In [27]:
await fresh(conn)


async def load(pairs):
    """Looks careful, and rolls nothing back."""
    for kind, number in pairs:
        await conn.execute("INSERT INTO tally VALUES ($1, $2)", kind, number)


try:
    await load([("click", 1), ("view", 2), ("purchase", "not a number")])
except exceptions.DataError:
    print("the load failed halfway")

print("and kept:", await tally(watcher))


the load failed halfway
and kept: [('click', 1), ('view', 2)]


Two rows written, one statement failed, nothing rolled back, and no exception anywhere says so. A
program ported from psycopg without touching its writes behaves like this everywhere, and the damage
shows up as half-finished data rather than as an error.

One line fixes the whole function, and it is the line to add first when porting:


In [28]:
await fresh(conn)


async def load_safely(pairs):
    async with conn.transaction():                                  # the whole loop, or none of it
        for kind, number in pairs:
            await conn.execute("INSERT INTO tally VALUES ($1, $2)", kind, number)


try:
    await load_safely([("click", 1), ("view", 2), ("purchase", "not a number")])
except exceptions.DataError:
    print("the load failed halfway")

print("and kept:", await tally(watcher))


the load failed halfway
and kept: []


In [29]:
for connection in (conn, watcher):
    await connection.close()
print("connections closed")


connections closed


## Recap

- asyncpg is PostgreSQL only, asynchronous only, and not a DB-API driver. Almost nothing from
  psycopg carries over except the SQL.
- Parameters are `$1`, `$2`, numbered, and may be repeated. A `%s` reaches the server as a `%`.
- `fetch` gives a list, `fetchrow` gives one `Record` or `None`, `fetchval` gives one value or
  `None`, and `execute` gives the command tag.
- A `Record` is a tuple and a mapping at once, and it is read-only. There is no row factory because
  there is nothing left to choose.
- A statement outside `async with conn.transaction()` is committed when it returns. There is no
  `conn.commit()`, and there is nothing to roll back.
- A nested block is a savepoint, `isolation=` sets the level with underscores, and every fetch method
  takes `timeout=` in seconds and raises a bare `TimeoutError`.
- A `jsonb` column arrives as text until you register a codec with `set_type_codec`.


## What is next

**Prepared Statements** is what asyncpg has been doing for every query in this notebook without
saying so, and the error that comes of it: a cached plan that an `ALTER TABLE` on another connection
made wrong.


---

&#8592; **Previous:** [AsyncConnection](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/10-async-connection.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
